# MNIST Digit Classification - Kata Tecnica Senior ML

Notebook oficial del reto:
- Entrenamiento de una CNN con Keras (sin modelos preentrenados)
- Evaluacion en conjunto de prueba
- Curvas de accuracy/loss
- Exportacion del modelo en `.h5` hacia `api/model/mnist_cnn.h5`

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

# Reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow: {tf.__version__}")

In [ ]:
# Carga y preprocesamiento
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

X_train = (X_train_raw.astype("float32") / 255.0)[..., np.newaxis]
X_test = (X_test_raw.astype("float32") / 255.0)[..., np.newaxis]

y_train = to_categorical(y_train_raw, 10)
y_test = to_categorical(y_test_raw, 10)

val_size = int(0.1 * len(X_train))
X_val, y_val = X_train[:val_size], y_train[:val_size]
X_tr, y_tr = X_train[val_size:], y_train[val_size:]

print("Train:", X_tr.shape, y_tr.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
# Modelo CNN
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
], name="mnist_cnn")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Callbacks y entrenamiento
os.makedirs("api/model", exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath="api/model/mnist_cnn_best.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Curvas de entrenamiento (accuracy / loss)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history.history["accuracy"]) + 1)
axes[0].plot(epochs, history.history["accuracy"], label="train")
axes[0].plot(epochs, history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history.history["loss"], label="train")
axes[1].plot(epochs, history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("api/model/training_curves.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Evaluacion
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\nClassification report:")
print(classification_report(y_test_raw, y_pred, target_names=[str(i) for i in range(10)]))

cm = confusion_matrix(y_test_raw, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de confusion")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig("api/model/confusion_matrix.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Exportacion final (.h5) alineada con la API
MODEL_PATH = "api/model/mnist_cnn.h5"
model.save(MODEL_PATH)

print("Modelo exportado en:", MODEL_PATH)
print("Existe?:", os.path.exists(MODEL_PATH))
print("Tamano (KB):", round(os.path.getsize(MODEL_PATH) / 1024, 1))

loaded = keras.models.load_model(MODEL_PATH)
loaded_acc = loaded.evaluate(X_test, y_test, verbose=0)[1]
print("Accuracy modelo cargado:", round(float(loaded_acc), 4))